In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x = x.float()                                   # fp32 reduction
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x * rms).to(dtype) * self.weight


class SwiGLU(nn.Module):
    def __init__(self, dim, mult=4, multiple_of=64):
        super().__init__()
        hidden = int(dim * mult * 2 / 3)
        hidden = multiple_of * ((hidden + multiple_of - 1) // multiple_of)

        self.w1 = nn.Linear(dim, hidden, bias=False)    # gate branch
        self.w2 = nn.Linear(dim, hidden, bias=False)    # value branch
        self.w3 = nn.Linear(hidden, dim, bias=False)    # project back

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


# ---------------- tests ----------------

torch.manual_seed(42)
B, T, D = 4, 32, 768
x = torch.randn(B, T, D)

norm = RMSNorm(D)
y = norm(x)
rms = y.pow(2).mean(-1).sqrt()
assert y.shape == x.shape
assert torch.allclose(rms, torch.ones_like(rms), atol=1e-4)
print(f"RMSNorm   ok | out RMS {rms.mean():.6f} | params {sum(p.numel() for p in norm.parameters()):,}")

# fp16 stability: large activations must not overflow
xb = torch.randn(2, 8, D) * 300
assert torch.isfinite(norm(xb.half() if torch.cuda.is_available() else xb)).all()
print("RMSNorm   fp16 overflow test ok")

mlp = SwiGLU(D)
z = mlp(x)
assert z.shape == x.shape
print(f"SwiGLU    ok | hidden {mlp.w1.out_features} | params {sum(p.numel() for p in mlp.parameters()):,}")

RMSNorm   ok | out RMS 1.000000 | params 768
RMSNorm   fp16 overflow test ok
SwiGLU    ok | hidden 2048 | params 4,718,592


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RoPE(nn.Module):
    """Contiguous-half rotary embeddings.
    x = [x1 | x2] -> [x1*cos - x2*sin, x2*cos + x1*sin]
    This convention MUST match the inference path exactly."""

    def __init__(self, head_dim, max_seq):
        super().__init__()
        assert head_dim % 2 == 0, "head_dim must be even for RoPE"
        self.head_dim, self.max_seq = head_dim, max_seq

        inv_freq = 1.0 / (10000 ** (
            torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_seq, dtype=torch.float32), inv_freq)

        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, x, pos0=0):
        """x: (B, H, T, head_dim). pos0 is the absolute position of x[..., 0, :],
        used by cached generation where T == 1."""
        dtype = x.dtype
        B, H, T, D = x.shape
        assert D == self.head_dim
        assert pos0 + T <= self.max_seq, f"position {pos0 + T} exceeds max_seq {self.max_seq}"

        half = D // 2
        x = x.float()
        x1, x2 = x[..., :half], x[..., half:]

        cos = self.cos[pos0:pos0 + T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[pos0:pos0 + T].unsqueeze(0).unsqueeze(0)

        return torch.cat([x1 * cos - x2 * sin,
                          x2 * cos + x1 * sin], dim=-1).to(dtype)


class CausalSelfAttention(nn.Module):
    def __init__(self, dim, n_heads, max_seq, rope=True):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"

        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.qkv      = nn.Linear(dim, 3 * dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope     = RoPE(self.head_dim, max_seq) if rope else None

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)   # (B, H, T, hd)

        if self.rope is not None:
            q, k = self.rope(q), self.rope(k)            # never v

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)


# ============================================================
# TESTS
# ============================================================

torch.manual_seed(42)
B, T, D, H = 2, 16, 768, 12

attn = CausalSelfAttention(D, H, max_seq=2048, rope=False)
x = torch.randn(B, T, D)
y = attn(x)
assert y.shape == x.shape and torch.isfinite(y).all()
print("attention  shape/finite ok")

# causality: perturbing position t must not affect positions < t
t = 7
x1 = torch.randn(B, T, D)
x2 = x1.clone()
x2[:, t, :] = torch.randn(B, D)
with torch.no_grad():
    y1, y2 = attn(x1), attn(x2)

assert torch.allclose(y1[:, :t], y2[:, :t], atol=1e-6), "LEAK: past positions changed"
assert not torch.allclose(y1[:, t:], y2[:, t:], atol=1e-6), "position t had no effect"
print("attention  causality ok")

# --- RoPE ---
hd, max_seq = 64, 2048
rope = RoPE(hd, max_seq)

q = torch.randn(2, 8, 32, hd)
assert torch.allclose(q.norm(dim=-1), rope(q).norm(dim=-1), atol=1e-5), "not a rotation"
print("rope       norm preservation ok")

# relative position: score depends only on (i - j), with DISTINCT q and k
qv, kv = torch.randn(hd), torch.randn(hd)
qt, kt = torch.zeros(1, 1, 16, hd), torch.zeros(1, 1, 16, hd)
qt[0, 0, 5], kt[0, 0, 3]   = qv, kv      # offset 2
qt[0, 0, 12], kt[0, 0, 10] = qv, kv      # offset 2
qr, kr = rope(qt), rope(kt)

d1 = (qr[0, 0, 5]  * kr[0, 0, 3]).sum()
d2 = (qr[0, 0, 12] * kr[0, 0, 10]).sum()
d3 = (qr[0, 0, 12] * kr[0, 0, 3]).sum()   # offset 9 — must differ

assert torch.allclose(d1, d2, atol=1e-5), f"not relative: {d1:.6f} vs {d2:.6f}"
assert not torch.allclose(d1, d3, atol=1e-3), "position has no effect at all"
print(f"rope       relative position ok  (off2 {d1:.4f}={d2:.4f}, off9 {d3:.4f})")

# pos0 offset must equal the same absolute position computed in one pass
full = rope(qt)
piece = rope(qt[:, :, 12:13], pos0=12)
assert torch.allclose(full[:, :, 12:13], piece, atol=1e-5), "pos0 offset broken"
print("rope       pos0 offset ok")

# --- combined ---
rattn = CausalSelfAttention(768, 12, max_seq=2048)
xr = torch.randn(2, 128, 768)
with torch.no_grad():
    yr = rattn(xr)
assert yr.shape == xr.shape and torch.isfinite(yr).all()
print(f"rope-attn  ok | params {sum(p.numel() for p in rattn.parameters()):,}")

attention  shape/finite ok
attention  causality ok
rope       norm preservation ok
rope       relative position ok  (off2 -0.3179=-0.3179, off9 -4.3299)
rope       pos0 offset ok
rope-attn  ok | params 2,359,296


In [3]:
import os, glob, json
from pathlib import Path
import numpy as np
from tokenizers import Tokenizer

DATA = Path("/kaggle/input/datasets/megs132/capstone-1")

for p in sorted(DATA.iterdir()):
    print(f"{p.stat().st_size/1e6:9.1f} MB  {p.name}")

WORK     = "/kaggle/working"
CKPT_DIR = f"{WORK}/ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)

TRAIN_BIN, VAL_BIN = DATA / "bpe32k_train.bin", DATA / "bpe32k_val.bin"
META_PATH, TOK_PATH = DATA / "bpe32k_meta.json", DATA / "bpe32k.json"
TEST_PATH = DATA / "test.json"
VAL_PROMPTS_PATH, VAL_REFS_PATH = DATA / "val_prompts.json", DATA / "val_refs.json"

for p in (TRAIN_BIN, VAL_BIN, META_PATH, TOK_PATH, TEST_PATH,
          VAL_PROMPTS_PATH, VAL_REFS_PATH):
    assert p.exists(), f"missing: {p}"

META = json.load(open(META_PATH))
VOCAB_SIZE = META["vocab_size"]
PAD_ID = META["special_token_ids"]["pad"]
EOS_ID = META["special_token_ids"]["eos"]

tok = Tokenizer.from_file(str(TOK_PATH))
train_bin = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_bin   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")

test_data   = json.load(open(TEST_PATH))
VAL_PROMPTS = json.load(open(VAL_PROMPTS_PATH))
VAL_REFS    = json.load(open(VAL_REFS_PATH))

print(f"\nvocab {VOCAB_SIZE:,} | pad {PAD_ID} | eos {EOS_ID}")
print(f"train {len(train_bin):,} tokens | val {len(val_bin):,} tokens")
print(f"test {len(test_data)} | val prompts {len(VAL_PROMPTS)}")
print("\nsample:", tok.decode(train_bin[:60].astype(np.int64).tolist())[:300])

      0.0 MB  __huggingface_repos__.json
      2.3 MB  bpe32k.json
      0.0 MB  bpe32k_meta.json
    434.4 MB  bpe32k_train.bin
      5.1 MB  bpe32k_val.bin
      0.0 MB  corpus_stats.json
      3.0 MB  test.json
    181.6 MB  train_acts.jsonl
    367.1 MB  train_high_courts.jsonl
    296.2 MB  train_supreme_court.jsonl
    183.4 MB  train_tribunals.jsonl
      3.1 MB  val_acts.jsonl
      3.1 MB  val_high_courts.jsonl
      0.9 MB  val_prompts.json
      0.2 MB  val_refs.json
      3.0 MB  val_supreme_court.jsonl
      3.1 MB  val_tribunals.jsonl

vocab 32,000 | pad 0 | eos 3
train 217,201,090 tokens | val 2,560,200 tokens
test 1000 | val prompts 300

sample: Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors inclu


In [4]:
import math, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB_SIZE = META["vocab_size"]
DIM, N_LAYERS, N_HEADS, MAX_SEQ = 768, 12, 12, 2048
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class TransformerBlock(nn.Module):
    def __init__(self, dim, n_heads, max_seq):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn  = CausalSelfAttention(dim, n_heads, max_seq)
        self.norm2 = RMSNorm(dim)
        self.mlp   = SwiGLU(dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class GPTModel(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, dim=DIM,
                 n_layers=N_LAYERS, n_heads=N_HEADS, max_seq=MAX_SEQ):
        super().__init__()
        self.vocab_size, self.dim = vocab_size, dim
        self.n_layers, self.n_heads, self.max_seq = n_layers, n_heads, max_seq

        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.blocks  = nn.ModuleList(
            [TransformerBlock(dim, n_heads, max_seq) for _ in range(n_layers)])
        self.norm_f  = RMSNorm(dim)
        self.head    = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight          # tie

        self.apply(self._init_weights)

        # rescale residual-writing projections AFTER the blanket init
        std = 0.02 / math.sqrt(2 * n_layers)
        for b in self.blocks:
            nn.init.normal_(b.attn.out_proj.weight, 0.0, std)
            nn.init.normal_(b.mlp.w3.weight,        0.0, std)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        T = idx.size(1)
        if T > self.max_seq:
            raise ValueError(f"T={T} exceeds max_seq={self.max_seq}")

        x = self.tok_emb(idx)
        for b in self.blocks:
            x = b(x)
        logits = self.head(self.norm_f(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size),
                                   targets.reshape(-1))
        return logits, loss


def count_params(m):
    """Dedupe by id() — tied head and embedding are one Parameter."""
    return sum(p.numel() for p in {id(p): p for p in m.parameters()}.values())


# ============================================================
torch.manual_seed(42)
model = GPTModel().to(DEVICE)

# --- 1. parameter count ---
total = count_params(model)
emb   = model.tok_emb.weight.numel()
blk   = sum(count_params(b) for b in model.blocks)

print(f"params {total:,} ({total/1e6:.2f}M)")
assert 50_000_000 <= total <= 300_000_000, f"{total:,} outside 50M-300M"
print(f"  embedding (tied) {emb:,}  ({100*emb/total:.1f}%)")
print(f"  blocks           {blk:,}  ({100*blk/total:.1f}%)")
print(f"  per block        {blk//N_LAYERS:,}")

# --- 2. initial loss ---
model.eval()
idx = torch.randint(0, VOCAB_SIZE, (2, 128), device=DEVICE)
tgt = torch.randint(0, VOCAB_SIZE, (2, 128), device=DEVICE)
with torch.no_grad():
    _, init_loss = model(idx, tgt)
init_loss = init_loss.item()

print(f"\ninitial loss {init_loss:.4f} vs ln(vocab) {math.log(VOCAB_SIZE):.4f}")
assert 9.0 < init_loss < 12.0, f"unexpected init loss {init_loss:.4f}"

# --- 3. end-to-end causality (relative criterion) ---
torch.manual_seed(123)
x1 = torch.randint(0, VOCAB_SIZE, (1, 64), device=DEVICE)
x2 = x1.clone()
t = 31
x2[0, t] = (x1[0, t].item() + 1) % VOCAB_SIZE

with torch.no_grad():
    y1, _ = model(x1)
    y2, _ = model(x2)

before = (y1[:, :t] - y2[:, :t]).abs().max().item()
after  = (y1[:, t:] - y2[:, t:]).abs().max().item()
print(f"\ncausality: max delta before t {before:.2e} | from t {after:.2e}")
assert before < after * 1e-4, "LEAK: future token affected past positions"
assert after > 1e-3, "perturbation had no effect"
print("causality ok")

# --- 4. overfit one real batch ---
train_mm = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
BS, SL = 2, 128
need = BS * (SL + 1)
chunk = torch.from_numpy(train_mm[:need].astype(np.int64)).to(DEVICE).view(BS, SL + 1)
bx, by = chunk[:, :-1], chunk[:, 1:]

torch.manual_seed(42)
om = GPTModel().to(DEVICE)
om.train()
opt = torch.optim.AdamW(om.parameters(), lr=1e-3)
amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp)

hist = []
for step in range(100):
    opt.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
        _, loss = om(bx, by)
    scaler.scale(loss).backward()
    scaler.step(opt); scaler.update()
    hist.append(loss.item())
    if step == 0 or (step + 1) % 20 == 0:
        print(f"  step {step+1:3d}/100  loss {loss.item():.4f}")

print(f"\noverfit {hist[0]:.4f} -> {hist[-1]:.4f}")
assert hist[-1] < 1.0, f"gradients not flowing: final loss {hist[-1]:.4f}"
print("overfit ok")

del om, opt, scaler
torch.cuda.empty_cache()
print(f"\nMODEL READY — {total/1e6:.2f}M params")

params 109,529,856 (109.53M)
  embedding (tied) 24,576,000  (22.4%)
  blocks           84,953,088  (77.6%)
  per block        7,079,424

initial loss 10.5504 vs ln(vocab) 10.3735

causality: max delta before t 0.00e+00 | from t 1.47e+00
causality ok
  step   1/100  loss 10.6202
  step  20/100  loss 3.8214
  step  40/100  loss 3.1543
  step  60/100  loss 2.3678
  step  80/100  loss 0.5886
  step 100/100  loss 0.0964

overfit 10.6202 -> 0.0964
overfit ok

MODEL READY — 109.53M params


In [5]:
import glob, torch
from pathlib import Path

hits = glob.glob("/kaggle/input/**/A_decoder.pt", recursive=True)
assert hits, "phase1-model-a not attached"
CKPT_A = Path(hits[0])
print("checkpoint:", CKPT_A, f"({CKPT_A.stat().st_size/1e6:.1f} MB)")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = GPTModel().to(DEVICE)
sd = torch.load(CKPT_A, map_location="cpu")
missing, unexpected = model.load_state_dict(
    {k: v.float() for k, v in sd.items()}, strict=True)
model.eval()

print(f"loaded {count_params(model)/1e6:.2f}M params")

checkpoint: /kaggle/input/datasets/megs132/phase1-model-a/A_decoder.pt (268.2 MB)
loaded 109.53M params


In [6]:
import json
ids = torch.tensor([tok.encode(test_data["1"]["text"]).ids], device=DEVICE)
print("prompt tokens:", ids.shape[1])

with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16,
                                     enabled=DEVICE == "cuda"):
    logits, _ = model(ids)

top = logits[0, -1].float().topk(10)
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"  {p:7.2f}  {tok.decode([i])!r}")

prompt tokens: 625
     8.59  ' well'
     8.46  ' a'
     8.31  ' not'
     7.82  ','
     7.74  ' true'
     7.49  ' the'
     7.34  ' trite'
     7.34  ' now'
     7.17  ' open'
     7.17  ' also'


In [7]:
import numpy as np
import torch
import torch.nn.functional as F

USE_AMP = DEVICE == "cuda"


def cached_attention(attn, x, cache=None, pos0=0):
    """One attention layer with KV cache. Returns (out, (k, v))."""
    B, T, C = x.shape

    qkv = attn.qkv(x).view(B, T, 3, attn.n_heads, attn.head_dim)
    q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)

    q = attn.rope(q, pos0=pos0)
    k = attn.rope(k, pos0=pos0)

    if cache is not None:
        k = torch.cat([cache[0], k], dim=2)
        v = torch.cat([cache[1], v], dim=2)

    # T>1 prefill needs the causal mask; T==1 attends over the whole
    # cache and must NOT be masked (is_causal would assume query pos 0).
    out = F.scaled_dot_product_attention(q, k, v, is_causal=(T > 1))
    out = out.transpose(1, 2).reshape(B, T, C)
    return attn.out_proj(out), (k, v)


def cached_forward(model, idx, caches=None, pos0=0):
    x = model.tok_emb(idx)
    new_caches = []

    for i, block in enumerate(model.blocks):
        a, c = cached_attention(block.attn, block.norm1(x),
                                None if caches is None else caches[i], pos0)
        x = x + a
        x = x + block.mlp(block.norm2(x))
        new_caches.append(c)

    x = model.norm_f(x)
    return model.head(x[:, -1:]), new_caches      # last position only


@torch.no_grad()
def prefill(model, prompt):
    logits, caches = cached_forward(model, prompt, caches=None, pos0=0)
    return logits, caches, prompt.shape[1]


@torch.no_grad()
def step(model, token, caches):
    pos0 = caches[0][0].shape[2]                  # absolute pos of this token
    return cached_forward(model, token, caches=caches, pos0=pos0)


# ============================================================
# EQUIVALENCE TEST
# ============================================================

model.eval()
SPLIT, N_STEPS = 494, 4

x = torch.from_numpy(np.asarray(train_bin[:500], dtype=np.int64)
                     ).unsqueeze(0).to(DEVICE)

with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
    full_logits, _ = model(x)

    # prefill on x[:, :494] ends AT position 493 -> predicts token 494
    ref = full_logits[:, SPLIT - 1: SPLIT - 1 + N_STEPS + 1]

    pl, caches, plen = prefill(model, x[:, :SPLIT])
    assert plen == SPLIT and caches[0][0].shape[2] == SPLIT

    got = [pl]
    for i in range(SPLIT, SPLIT + N_STEPS):
        lg, caches = step(model, x[:, i:i + 1], caches)
        assert caches[0][0].shape[2] == i + 1, "cache length drifted"
        got.append(lg)
    got = torch.cat(got, dim=1)

assert got.shape == ref.shape, f"{got.shape} vs {ref.shape}"

diff  = (got.float() - ref.float()).abs()
agree = (got.argmax(-1) == ref.argmax(-1)).float().mean().item() * 100

print("=" * 60)
print("CACHED INFERENCE EQUIVALENCE")
print("=" * 60)
print(f"positions compared : {SPLIT-1} .. {SPLIT-1+N_STEPS}  (prefill + {N_STEPS} steps)")
print(f"max abs diff       : {diff.max().item():.6f}")
print(f"mean abs diff      : {diff.mean().item():.6f}")
print(f"per-position max   : {[round(d, 4) for d in diff.amax(-1)[0].tolist()]}")
print(f"top-1 agreement    : {agree:.1f}%")
print()
print("PASS" if diff.max().item() < 0.05 and agree == 100.0 else "FAIL")

CACHED INFERENCE EQUIVALENCE
positions compared : 493 .. 497  (prefill + 4 steps)
max abs diff       : 0.007812
mean abs diff      : 0.000488
per-position max   : [0.0039, 0.0078, 0.0078, 0.0078, 0.0039]
top-1 agreement    : 100.0%

PASS


In [8]:
import torch
import torch.nn.functional as F

NEG = -1e4          # finite: -inf can make a fully-masked row NaN in fp16

SPECIAL_TOKEN_IDS = [i for i in (
    tok.token_to_id("<|pad|>"), tok.token_to_id("<|unk|>"),
    tok.token_to_id("<|bos|>"), tok.token_to_id("<|eos|>")) if i is not None]
print("banned ids:", SPECIAL_TOKEN_IDS)


@torch.no_grad()
def sample_next_token(logits, history, ngram_maps=None,
                      temperature=1.0, top_k=None, top_p=None,
                      repetition_penalty=1.0, repetition_window=128,
                      no_repeat_ngram_size=None):
    """logits (B, vocab); history (B, T) = prompt + generated so far.
    ngram_maps: list of dicts, one per row, carried across calls."""
    logits = logits.float().clone()
    logits[:, SPECIAL_TOKEN_IDS] = NEG

    # --- repetition penalty over recent history (prompt included) ---
    if repetition_penalty != 1.0 and history.shape[1] > 0:
        recent = history[:, -repetition_window:]
        gathered = torch.gather(logits, 1, recent)
        gathered = torch.where(gathered > 0,
                               gathered / repetition_penalty,
                               gathered * repetition_penalty)
        logits.scatter_(1, recent, gathered)

    # --- no-repeat n-gram, incremental ---
    n = no_repeat_ngram_size or 0
    if n > 1 and ngram_maps is not None:
        for b in range(logits.shape[0]):
            seq = history[b].tolist()
            if len(seq) >= n - 1:
                banned = ngram_maps[b].get(tuple(seq[-(n - 1):]))
                if banned:
                    logits[b, list(banned)] = NEG

    if temperature <= 0:
        return logits.argmax(-1, keepdim=True)

    logits = logits / temperature

    if top_k:
        k = min(top_k, logits.shape[-1])
        kth = logits.topk(k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, NEG)

    if top_p is not None and top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cum = sl.softmax(-1).cumsum(-1)
        drop = cum > top_p
        drop[:, 1:] = drop[:, :-1].clone()      # keep the token that crosses p
        drop[:, 0] = False
        logits = logits.masked_fill(
            torch.zeros_like(drop).scatter_(1, si, drop), NEG)

    probs = logits.softmax(-1)
    bad = ~torch.isfinite(probs).all(-1) | (probs.sum(-1) <= 0)
    if bad.any():                                # degenerate row -> greedy
        probs[bad] = 0.0
        probs[bad, logits[bad].argmax(-1)] = 1.0
    return torch.multinomial(probs, 1)


def update_ngram_maps(ngram_maps, history, n):
    """Call after appending each new token. Records the ngram just completed."""
    if n > 1:
        for b in range(history.shape[0]):
            seq = history[b].tolist()
            if len(seq) >= n:
                ngram_maps[b].setdefault(tuple(seq[-n:-1]), set()).add(seq[-1])

banned ids: [0, 1, 2, 3]


In [11]:
torch.manual_seed(0)
lg = torch.randn(4, VOCAB_SIZE, device=DEVICE)
hist = torch.randint(0, VOCAB_SIZE, (4, 50), device=DEVICE)

g = sample_next_token(lg, hist, temperature=0)
assert torch.equal(g.squeeze(1), lg.argmax(-1)), "greedy broken"

s = sample_next_token(lg, hist, temperature=0.7, top_k=50, top_p=0.9,
                      repetition_penalty=1.1)
assert s.shape == (4, 1) and torch.isfinite(s.float()).all()
assert not any(t in SPECIAL_TOKEN_IDS for t in s.flatten().tolist())

tight = sample_next_token(lg, hist, temperature=0.7, top_k=1)
assert torch.equal(tight.squeeze(1), lg.argmax(-1)), "top_k=1 should be greedy"
print("sampler ok")

sampler ok


In [9]:
import numpy as np
import torch

PAD_ID = int(tok.token_to_id("<|pad|>"))

BATCH_SIZE, MAX_NEW_TOKENS, TARGET_WORDS, CHECK_EVERY = 32, 250, 128, 16


@torch.no_grad()
def generate_batched(test_data, batch_size=BATCH_SIZE,
                     max_new_tokens=MAX_NEW_TOKENS, target_words=TARGET_WORDS,
                     check_every=CHECK_EVERY, temperature=0.7, top_p=0.9,
                     top_k=50, repetition_penalty=1.1, repetition_window=128,
                     no_repeat_ngram_size=None, verbose=True):
    model.eval()

    items = []
    for key, value in test_data.items():
        text = value["text"] if isinstance(value, dict) else value
        ids = tok.encode(text).ids
        items.append(dict(id=str(key), text=text, ids=ids, length=len(ids)))
    items.sort(key=lambda d: d["length"])

    real = sum(d["length"] for d in items)
    padded = sum(max(d["length"] for d in items[s:s + batch_size]) * len(items[s:s + batch_size])
                 for s in range(0, len(items), batch_size))
    if verbose:
        print(f"padding waste (sorted): {100*(padded-real)/padded:.2f}%")

    results, n_batches = {}, (len(items) + batch_size - 1) // batch_size

    for bi, start in enumerate(range(0, len(items), batch_size), 1):
        batch = items[start:start + batch_size]
        B = len(batch)
        max_len = max(d["length"] for d in batch)

        prompt = torch.full((B, max_len), PAD_ID, dtype=torch.long)
        for b, d in enumerate(batch):                       # LEFT pad
            prompt[b, max_len - d["length"]:] = torch.tensor(d["ids"], dtype=torch.long)
        prompt = prompt.to(DEVICE)

        history = prompt.clone()
        ngram_maps = [dict() for _ in range(B)]
        finished = torch.zeros(B, dtype=torch.bool, device=DEVICE)
        generated = [[] for _ in range(B)]

        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits, caches, _ = prefill(model, prompt)

        for gstep in range(max_new_tokens):
            nxt = sample_next_token(
                logits[:, -1, :], history, ngram_maps,
                temperature=temperature, top_k=top_k, top_p=top_p,
                repetition_penalty=repetition_penalty,
                repetition_window=repetition_window,
                no_repeat_ngram_size=no_repeat_ngram_size)

            nxt = torch.where(finished.unsqueeze(1),
                              torch.full_like(nxt, PAD_ID), nxt)
            history = torch.cat([history, nxt], dim=1)

            if no_repeat_ngram_size:
                update_ngram_maps(ngram_maps, history, no_repeat_ngram_size)

            done_now = finished.tolist()
            for b, t in enumerate(nxt[:, 0].tolist()):
                if not done_now[b]:
                    generated[b].append(t)

            if (gstep + 1) % check_every == 0 or gstep + 1 == max_new_tokens:
                for b in range(B):
                    if not finished[b] and len(tok.decode(generated[b]).split()) >= target_words:
                        finished[b] = True
                if bool(finished.all()):
                    break

            if gstep + 1 < max_new_tokens:
                with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                    logits, caches = step(model, nxt, caches)

        for b, d in enumerate(batch):
            results[d["id"]] = dict(prompt=d["text"], generated_ids=generated[b])

        del caches, history, prompt, logits
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

        if verbose:
            print(f"  batch {bi}/{n_batches} | prompt_len {max_len} | "
                  f"gen {max(len(g) for g in generated)} tok", flush=True)

    return {str(k): results[str(k)] for k in test_data}        # restore order


def postprocess_predictions(results, test_data, target_words=128):
    preds, short = {}, []

    for key in test_data:
        key = str(key)
        words = tok.decode(results[key]["generated_ids"]).split()
        if len(words) < target_words:
            short.append((key, len(words)))
        preds[key] = {"text": " ".join(words[:target_words])}

    assert set(preds) == {str(k) for k in test_data}, "key set mismatch"

    for key, p in preds.items():
        pt = test_data[key]["text"] if isinstance(test_data[key], dict) else test_data[key]
        assert not p["text"].startswith(pt[:80]), f"echoes prompt: {key}"

    print(f"postprocess | short: {len(short)}/{len(preds)}"
          + (f" -> {short[:5]}" if short else ""))
    return preds, short


# ---------------- 5-prompt smoke test ----------------
five = {str(k): test_data[k] for k in list(test_data)[:5]}

res5 = generate_batched(five, batch_size=5, temperature=0.7, top_p=0.9,
                        top_k=50, repetition_penalty=1.1)
pred5, short5 = postprocess_predictions(res5, five)

for key in list(pred5)[:3]:
    print(f"\n{'='*70}\nID {key}")
    print("\nPROMPT TAIL:", (five[key]["text"])[-300:])
    print("\nGENERATED:", pred5[key]["text"])
    print(f"\nwords: {len(pred5[key]['text'].split())}")

padding waste (sorted): 3.95%
  batch 1/1 | prompt_len 643 | gen 192 tok
postprocess | short: 0/5

ID 1

PROMPT TAIL: dings need not be set at naught if the party has not approached the Court within reasonable time, as judicial review is not permissible at a belated stage. This Court held as under: " Delay in challenging the notification was fatal and writ petition entails with dismissal on grounds of laches. It is

GENERATED: well-settled that when a person is deprived of his property or his right to life cannot be taken away by a subsequent declaration by the court as it would amount to a violation of Article 14 of the Constitution of India." In State of Rajasthan & Anr. Vs. S.P. Gurcharan Singh (1995) 4 SCC 1 this Court again observed as under: "We have carefully considered the submissions made by learned counsel for the parties and have perused the material available on record. We are of the view that the High Court was not right in dismissing the writ petition on the ground of del

In [13]:
import time
t0 = time.time()
results = generate_batched(test_data, batch_size=32,
                           temperature=0.7, top_p=0.9, top_k=50,
                           repetition_penalty=1.1)
preds, short = postprocess_predictions(results, test_data)
print(f"\n{(time.time()-t0)/60:.1f} min")

padding waste (sorted): 0.74%
  batch 1/32 | prompt_len 563 | gen 176 tok
  batch 2/32 | prompt_len 568 | gen 240 tok
  batch 3/32 | prompt_len 574 | gen 250 tok
  batch 4/32 | prompt_len 578 | gen 192 tok
  batch 5/32 | prompt_len 581 | gen 240 tok
  batch 6/32 | prompt_len 585 | gen 192 tok
  batch 7/32 | prompt_len 587 | gen 224 tok
  batch 8/32 | prompt_len 590 | gen 250 tok
  batch 9/32 | prompt_len 592 | gen 208 tok
  batch 10/32 | prompt_len 594 | gen 208 tok
  batch 11/32 | prompt_len 596 | gen 250 tok
  batch 12/32 | prompt_len 598 | gen 250 tok
  batch 13/32 | prompt_len 601 | gen 250 tok
  batch 14/32 | prompt_len 603 | gen 208 tok
  batch 15/32 | prompt_len 606 | gen 192 tok
  batch 16/32 | prompt_len 609 | gen 250 tok
  batch 17/32 | prompt_len 611 | gen 250 tok
  batch 18/32 | prompt_len 614 | gen 224 tok
  batch 19/32 | prompt_len 616 | gen 192 tok
  batch 20/32 | prompt_len 620 | gen 250 tok
  batch 21/32 | prompt_len 623 | gen 192 tok
  batch 22/32 | prompt_len 625 | g

In [15]:
import json, zipfile, os

os.makedirs("/kaggle/working/submissions", exist_ok=True)
jp = "/kaggle/working/submissions/prediction.json"
zp = "/kaggle/working/submissions/A_decoder_t07.zip"

json.dump(preds, open(jp, "w"), ensure_ascii=False)
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(jp, arcname="prediction.json")      # MUST be at archive root

with zipfile.ZipFile(zp) as z:
    assert z.namelist() == ["prediction.json"], z.namelist()
    p = json.loads(z.read("prediction.json"))

assert set(p) == set(test_data), "id mismatch"
short_rows = [(k, len(v["text"].split())) for k, v in p.items()
              if len(v["text"].split()) != 128]
assert all(n <= 128 for _, n in short_rows), "over-length output"
print(f"OK — {len(p)} ids, {len(short_rows)} short, "
      f"min {min(n for _, n in short_rows) if short_rows else 128} words, "
      f"{os.path.getsize(zp)/1e6:.2f} MB")

OK — 1000 ids, 15 short, min 13 words, 0.22 MB


In [10]:
!pip -q install rouge-score sacrebleu bert-score

import json, os, zipfile
import numpy as np
import pandas as pd
from rouge_score import rouge_scorer
import sacrebleu
from bert_score import BERTScorer

# BERTScore's roberta-large is an evaluation backbone only — it mirrors the
# official metric and is never part of the trained model or the submission.
SCORER = BERTScorer(model_type="roberta-large", lang="en",
                    rescale_with_baseline=False, device=DEVICE, batch_size=32)
ROUGE = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

assert set(VAL_PROMPTS) == set(VAL_REFS)
print(f"{len(VAL_PROMPTS)} validation prompts")


def score_config(cfg, prompts=None, refs=None, batch_size=32, max_new_tokens=350):
    prompts = prompts or VAL_PROMPTS
    refs    = refs or VAL_REFS
    print(f"\n{'='*70}\n{cfg['name']}\n{'='*70}", flush=True)

    gen = generate_batched(
        prompts, batch_size=batch_size, max_new_tokens=max_new_tokens,
        target_words=128, check_every=16,
        temperature=cfg["temperature"], top_p=cfg.get("top_p", 0.9),
        top_k=cfg.get("top_k", 50),
        repetition_penalty=cfg.get("repetition_penalty", 1.0),
        repetition_window=cfg.get("repetition_window", 128),
        no_repeat_ngram_size=cfg.get("no_repeat_ngram_size"),
        verbose=False)

    preds_d, short = postprocess_predictions(gen, prompts, target_words=128)

    keys  = list(prompts)
    preds = [preds_d[k]["text"] for k in keys]
    refs_t = [refs[k]["text"] for k in keys]
    srcs   = [refs[k].get("src", "unknown") for k in keys]

    _, _, f1 = SCORER.score(preds, refs_t)
    f1 = f1.cpu().numpy()

    rl = np.array([ROUGE.score(r, p)["rougeL"].fmeasure
                   for p, r in zip(preds, refs_t)])
    bleu = sacrebleu.corpus_bleu(preds, [refs_t]).score / 100.0

    by_src = {s: dict(n=int((np.array(srcs) == s).sum()),
                      f1=float(f1[np.array(srcs) == s].mean()))
              for s in sorted(set(srcs))}

    print(f"  BERTScore F1 {f1.mean():.4f} | ROUGE-L {rl.mean():.4f} | "
          f"BLEU-4 {bleu:.4f} | short {len(short)}/{len(preds)}")
    for s, v in by_src.items():
        print(f"    {s:16s} n={v['n']:3d}  F1={v['f1']:.4f}")

    return dict(name=cfg["name"], bert_f1=float(f1.mean()),
                rouge_l=float(rl.mean()), bleu4=bleu,
                short=len(short), by_src=by_src, cfg=cfg)


CONFIGS = [
    dict(name="t0.7",       temperature=0.7, top_p=0.9, top_k=50, repetition_penalty=1.1),
    dict(name="t0.7_ng4",   temperature=0.7, top_p=0.9, top_k=50, repetition_penalty=1.0,
         no_repeat_ngram_size=4),
    dict(name="t0.7_rp1.2", temperature=0.7, top_p=0.9, top_k=50, repetition_penalty=1.2),
    dict(name="t0.3",       temperature=0.3, top_p=0.9, top_k=50, repetition_penalty=1.1),
    dict(name="greedy_ng3", temperature=0.0, top_p=1.0, top_k=None, repetition_penalty=1.0,
         no_repeat_ngram_size=3),
    dict(name="t0.9",       temperature=0.9, top_p=0.9, top_k=50, repetition_penalty=1.1),
]

results = [score_config(c) for c in CONFIGS]

table = pd.DataFrame([{k: r[k] for k in
                       ("name", "bert_f1", "rouge_l", "bleu4", "short")}
                      for r in results]).sort_values("bert_f1", ascending=False)
print("\n" + "=" * 70 + "\nDECODING SWEEP\n" + "=" * 70)
print(table.to_string(index=False))

best = max(results, key=lambda r: r["bert_f1"])
print(f"\nper-source ({best['name']}):")
print(pd.DataFrame([dict(source=s, n=v["n"], bert_f1=v["f1"])
                    for s, v in best["by_src"].items()]
                   ).sort_values("bert_f1", ascending=False).to_string(index=False))


# ---------- top two -> full test set -> zips ----------
top2 = [r["cfg"] for r in sorted(results, key=lambda r: -r["bert_f1"])[:2]]
print("\ntop two:", [c["name"] for c in top2])

os.makedirs("/kaggle/working/submissions", exist_ok=True)

for cfg in top2:
    print(f"\n--- full test: {cfg['name']} ---", flush=True)
    gen = generate_batched(
        test_data, batch_size=32, max_new_tokens=350, target_words=128,
        temperature=cfg["temperature"], top_p=cfg.get("top_p", 0.9),
        top_k=cfg.get("top_k", 50),
        repetition_penalty=cfg.get("repetition_penalty", 1.0),
        repetition_window=cfg.get("repetition_window", 128),
        no_repeat_ngram_size=cfg.get("no_repeat_ngram_size"),
        verbose=False)

    preds, short = postprocess_predictions(gen, test_data, target_words=128)
    assert set(preds) == {str(k) for k in test_data}, "key mismatch"
    assert all(len(v["text"].split()) <= 128 for v in preds.values()), "over-length"

    jp = f"/kaggle/working/submissions/prediction_{cfg['name']}.json"
    zp = f"/kaggle/working/submissions/A_{cfg['name']}.zip"
    json.dump(preds, open(jp, "w"), ensure_ascii=False)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")

    print(f"  {zp} | {len(short)} short | {os.path.getsize(zp)/1e6:.2f} MB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.2 MB/s eta 0:00:00


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


300 validation prompts

t0.7
postprocess | short: 2/300 -> [('210', 107), ('273', 39)]
  BERTScore F1 0.8539 | ROUGE-L 0.2990 | BLEU-4 0.2055 | short 2/300
    acts             n= 47  F1=0.8346
    high_courts      n= 92  F1=0.8962
    supreme_court    n= 88  F1=0.8341
    tribunals        n= 73  F1=0.8369

t0.7_ng4
postprocess | short: 0/300
  BERTScore F1 0.8523 | ROUGE-L 0.2924 | BLEU-4 0.1940 | short 0/300
    acts             n= 47  F1=0.8406
    high_courts      n= 92  F1=0.8859
    supreme_court    n= 88  F1=0.8371
    tribunals        n= 73  F1=0.8358

t0.7_rp1.2
postprocess | short: 0/300
  BERTScore F1 0.8504 | ROUGE-L 0.2720 | BLEU-4 0.1969 | short 0/300
    acts             n= 47  F1=0.8359
    high_courts      n= 92  F1=0.8869
    supreme_court    n= 88  F1=0.8309
    tribunals        n= 73  F1=0.8371

t0.3
postprocess | short: 4/300 -> [('62', 68), ('223', 127), ('273', 47), ('296', 115)]
  BERTScore F1 0.8482 | ROUGE-L 0.2953 | BLEU-4 0.2043 | short 4/300
    acts       